# 🌿 DenseNet-121 + SVM
**Multicrop Disease Classification**

This notebook trains a **DenseNet-121** backbone, extracts **1024-d feature vectors**, and trains an **SVM classifier** on top using 5-fold cross-validation. Soft-voting ensemble is used at test time.

| Setting | Value |
|---|---|
| Backbone | DenseNet-121 (ImageNet pretrained) |
| Classifier | SVM (RBF kernel, probability=True) |
| Feature dim | 1024 (global avg pool output) |
| Optimizer | AdamW |
| Learning Rate | 0.001 |
| LR Scheduler | StepLR (step=10, γ=0.1) |
| Batch Size | 32 |
| Epochs | 50 (early stop patience=10) |
| K-Folds | 5 |
| Feature scaling | StandardScaler (per fold) |

> ⏱️ **Estimated time**: ~8–13 hours total on T4 | ~2.5–5 hours on A100  
> &nbsp;&nbsp;&nbsp;• Backbone training: ~7–12 hr  
> &nbsp;&nbsp;&nbsp;• SVM training: ~30–60 min (CPU)

> 💾 **Checkpoint resuming** is built-in — re-run from the training cell after a session reset.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Check GPU

In [ ]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3. Clone / Pull Repository

In [ ]:
import os

REPO_URL = 'https://github.com/RizStillLearning/multicrop_disease_classification.git'
REPO_DIR = '/content/multicrop_disease_classification'

if os.path.exists(REPO_DIR):
    print('Repo already exists — pulling latest changes...')
    %cd {REPO_DIR}
    !git pull
else:
    print('Cloning repository...')
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

print(f'Working directory: {os.getcwd()}')

## 4. Install Dependencies

In [ ]:
!pip install -q torchvision pytorch-grad-cam pyyaml seaborn tqdm joblib
print('✅ Dependencies installed')

## 5. Upload / Link Dataset

In [ ]:
# ── OPTION A: Link dataset from Google Drive (recommended) ────────────────────
DRIVE_DATASET_PATH = '/content/drive/MyDrive/datasets/multicrop_disease'

if os.path.exists(DRIVE_DATASET_PATH):
    !ln -sfn {DRIVE_DATASET_PATH} /content/multicrop_disease_classification/data
    print(f'✅ Dataset linked from Drive: {DRIVE_DATASET_PATH}')
else:
    print(f'⚠️  Drive path not found: {DRIVE_DATASET_PATH}')
    print('   Edit DRIVE_DATASET_PATH above, or use Option B below.')

In [ ]:
# ── OPTION B: Download from Kaggle ───────────────────────────────────────────
# !pip install -q kaggle
# from google.colab import files
# files.upload()   # upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d <dataset-slug> -p data/ --unzip

In [ ]:
# Verify dataset
data_dir = '/content/multicrop_disease_classification/data'
if os.path.exists(data_dir):
    classes = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    print(f'✅ Dataset found — {len(classes)} classes:')
    for c in sorted(classes):
        n = len(os.listdir(os.path.join(data_dir, c)))
        print(f'   {c}: {n} images')
else:
    print('❌ data/ directory not found.')

## 6. Link Output Folders to Google Drive

In [ ]:
DRIVE_OUTPUT = '/content/drive/MyDrive/multicrop_outputs/densenet121_svm'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

subdirs = [
    'densenet121_svm/backbone',
    'densenet121_svm/checkpoints',
    'densenet121_svm/classifier',
    'densenet121_svm/logs',
    'densenet121_svm/fold_results',
    'densenet121_svm/classification_reports',
    'densenet121_svm/plots',
]

for subdir in subdirs:
    drive_path = os.path.join(DRIVE_OUTPUT, subdir.split('/')[-1])
    local_path = f'/content/multicrop_disease_classification/{subdir}'
    os.makedirs(drive_path, exist_ok=True)
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    if not os.path.exists(local_path):
        !ln -s {drive_path} {local_path}

print('✅ Output directories linked to Google Drive')

## 7. Train Backbone (5-Fold CV)

> ⏱️ **Expected time per fold on T4**: ~1.5–2.5 hours  
> ⏱️ **Total (5 folds)**: ~7–12 hours  
> 🔁 Re-run after session reset — automatically resumes from last checkpoint.

In [ ]:
%cd /content/multicrop_disease_classification
!python densenet121_svm/scripts/train_backbone.py

## 8. Train SVM Classifier (5-Fold CV)

> ⏱️ **Expected time**: ~30–60 minutes (CPU-bound)  
> 📌 Run this only after backbone training (Step 7) is complete for all 5 folds.

In [ ]:
# Verify all backbone folds exist before training SVM
backbone_dir = 'densenet121_svm/backbone'
k_fold = 5
missing = [f'backbone_fold_{i+1}.pth' for i in range(k_fold)
           if not os.path.exists(os.path.join(backbone_dir, f'backbone_fold_{i+1}.pth'))]

if missing:
    print(f'❌ Missing backbone folds: {missing}')
    print('   Complete backbone training (Step 7) first.')
else:
    print(f'✅ All {k_fold} backbone folds found. Starting SVM training...')

In [ ]:
%cd /content/multicrop_disease_classification
!python densenet121_svm/scripts/train_SVM.py

## 9. View Training Logs

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_path = 'densenet121_svm/logs/training_log.csv'
if os.path.exists(log_path):
    df = pd.read_csv(log_path)
    print(df.tail(20).to_string(index=False))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(df['Epoch'], df['Train Loss'], label='Train Loss', color='royalblue')
    ax1.plot(df['Epoch'], df['Validation Loss'], label='Val Loss', color='tomato')
    ax1.set_title('Loss Curve'); ax1.legend(); ax1.set_xlabel('Epoch')

    ax2.plot(df['Epoch'], df['Validation Accuracy'], color='seagreen', label='Val Accuracy')
    ax2.set_title('Validation Accuracy'); ax2.legend(); ax2.set_xlabel('Epoch')
    plt.tight_layout()
    plt.savefig('densenet121_svm/plots/training_curves.png', dpi=120)
    plt.show()
else:
    print('No log file yet.')

## 10. View Fold Results

In [ ]:
for fold_file, label in [
    ('densenet121_svm/fold_results/backbone_fold_results.csv', 'Backbone'),
    ('densenet121_svm/fold_results/svm_fold_results.csv', 'SVM'),
]:
    if os.path.exists(fold_file):
        print(f'\n── {label} Fold Results ──')
        print(pd.read_csv(fold_file).to_string(index=False))
    else:
        print(f'{label} fold results not yet available.')

## 11. Visualize Attention Maps (GradCAM)

In [ ]:
%cd /content/multicrop_disease_classification
!python densenet121_svm/scripts/attention_map.py

from IPython.display import Image as IPImage
attn_path = 'densenet121_svm/plots/ensemble_attention_maps.png'
if os.path.exists(attn_path):
    display(IPImage(attn_path))

## 12. View Confusion Matrices

In [ ]:
from IPython.display import Image as IPImage
for cm_file, label in [
    ('densenet121_svm/plots/backbone_confusion_matrix.png', 'Backbone'),
    ('densenet121_svm/plots/svm_confusion_matrix.png', 'SVM'),
]:
    if os.path.exists(cm_file):
        print(f'\n── {label} Confusion Matrix ──')
        display(IPImage(cm_file, width=800))
    else:
        print(f'{label} confusion matrix not available yet.')

## 13. View Classification Reports

In [ ]:
import json
for report_file, label in [
    ('densenet121_svm/classification_reports/backbone_classification_report.json', 'Backbone'),
    ('densenet121_svm/classification_reports/svm_classification_report.json', 'SVM'),
]:
    if os.path.exists(report_file):
        print(f'\n── {label} Classification Report ──')
        with open(report_file) as f:
            report = json.load(f)
        df_report = pd.DataFrame(report).T.round(4)
        print(df_report.to_string())
    else:
        print(f'{label} report not available yet.')